# Pathway 2: Multi-Solid Global Assembly

This tutorial demonstrates assembling multiple geometric parts into a single domain and solving the fused system globally.

```
Multi Solid Model  →  FDS for Entire Domain  →  Reduced Order Model
```

## 1. Create a Project and Assembly

The `EMProject` manages the simulation lifecycle. An `Assembly` groups multiple parts along a principal axis.

In [ ]:
from core.em_project import EMProject
from geometry.primitives import RectangularWaveguide

proj = EMProject(name='pathway2_global', base_dir='./tutorial_sims')
assembly = proj.create_assembly(main_axis='Z')
print(f"Project: {proj.name}")

## 2. Add Components

We add two rectangular waveguide sections. The `after` argument auto-aligns the second component flush against the first.

In [ ]:
wg1 = RectangularWaveguide(a=0.1, L=0.1)
wg2 = RectangularWaveguide(a=0.1, L=0.1)

assembly.add("input", wg1)
assembly.add("output", wg2, after="input")
print(f"Components: {list(assembly.parts.keys())}")

## 3. Build and Mesh

The `build()` step fuses all parts into a **single OCC solid**. The interface between wg1 and wg2 disappears — it becomes one continuous domain.

In [ ]:
assembly.build()
assembly.generate_mesh(maxh=0.02)
assembly.show('mesh')

## 4. Solve the Full-Order Model

Since the assembly is fused, the solver sees a single mesh — identical to Pathway 1.

In [ ]:
results = proj.fds.solve(fmin=1.0, fmax=4.0, nsamples=50, store_snapshots=True)
print(f"Solved at {len(proj.fds.frequencies)} frequency points")
print(f"External ports: {proj.fds.ports}")

## 5. Reduce and Solve ROM

In [ ]:
import time

rom = proj.fds.fom.reduce(tol=1e-6)
rom.print_info()

t0 = time.time()
rom.solve(fmin=1.0, fmax=4.0, nsamples=500)
print(f"ROM solve time: {(time.time() - t0)*1e3:.1f} ms")

## 6. Plot Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
freq_ghz_fom = proj.fds.frequencies / 1e9
freq_ghz_rom = rom.frequencies / 1e9

ax = axes[0]
ax.plot(freq_ghz_fom, proj.fds.get_s_db(1, 1), 'o', label='FOM', ms=4)
ax.plot(freq_ghz_rom, rom.get_s_db(1, 1), '--', label='ROM', lw=1.5)
ax.set_xlabel('Frequency (GHz)'); ax.set_ylabel('|S11| (dB)')
ax.set_title('S11'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(freq_ghz_fom, proj.fds.get_s_db(1, 2), 'o', label='FOM', ms=4)
ax.plot(freq_ghz_rom, rom.get_s_db(1, 2), '--', label='ROM', lw=1.5)
ax.set_xlabel('Frequency (GHz)'); ax.set_ylabel('|S21| (dB)')
ax.set_title('S21'); ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Global Assembly: FOM vs ROM', fontsize=14)
plt.tight_layout(); plt.show()

## 7. Save the Project

The project saves the fused assembly, mesh, and all solver results.

In [ ]:
proj.save()
print(f"Project saved to: {proj.project_dir}")

## Summary

In this tutorial we:
1. Created a multi-part assembly with automatic alignment
2. Fused them into a single domain via `build()`
3. Solved using the standard FDS → ROM pipeline

This pathway is ideal for small assemblies. For larger systems, see:
- [Pathway 3: FOM Concatenation](pathway3_fom_concatenation.ipynb)
- [Pathway 4: ROM Concatenation](pathway4_rom_concatenation.ipynb)